# TaNP Gating-FiLM: три датасета на Colab T4
Ноутбук подготавливает данные и последовательно обучает отдельную модель для LastFM, MovieLens-1M и Gowalla. Все вычисления запускаются через `run_experiment.py`.

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/vladimirlavrentyev/cu-cold-start-recommendation-research.git'
PROJECT = Path('/content/cu-cold-start-recommendation-research')
if PROJECT.exists():
    !git -C {PROJECT} pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}

## Постоянное хранилище
Checkpoints, логи, таблицы и графики будут записываться прямо в Google Drive. Завершённые датасеты сохранятся даже при перезапуске Colab.

In [ ]:
from google.colab import drive
import json
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/tanp_gating_film_seed2020')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
with open(PROJECT / 'configs/tanp_gating_film_colab.json', encoding='utf-8') as stream:
    drive_config = json.load(stream)
drive_config['common']['output_dir'] = str(DRIVE_ROOT / 'artifacts/runs')
drive_config['common']['results_dir'] = str(DRIVE_ROOT / 'results')
drive_config['common']['figures_dir'] = str(DRIVE_ROOT / 'figures')
DRIVE_CONFIG = Path('/content/tanp_gating_film_colab_drive.json')
DRIVE_CONFIG.write_text(json.dumps(drive_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Результаты будут сохранены в:', DRIVE_ROOT)

In [ ]:
import torch
assert torch.cuda.is_available(), 'В Colab выберите Runtime → Change runtime type → T4 GPU'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
!python -m pip install -q -r requirements-colab.txt

## Подготовка данных
LastFM извлекается из авторского архива. MovieLens-1M загружается с GroupLens. Gowalla строится детерминированно из NGCF; точный подготовленный сплит Gowalla авторы TaNP не опубликовали, поэтому это реконструкция протокола.

In [ ]:
!python scripts/prepare_datasets.py --dataset all --seed 2020 --gowalla-max-users 2692

In [ ]:
!python run_experiment.py --suite {DRIVE_CONFIG} --dry-run

## Обучение
Следующая ячейка обучает три модели последовательно. Checkpoint сохраняется на Google Drive каждые 5 эпох. При повторном запуске `--skip-existing` пропускает завершённые датасеты, а `--resume` продолжает незавершённый датасет с последней сохранённой эпохи.

In [ ]:
!python run_experiment.py --suite {DRIVE_CONFIG} --skip-existing --resume

## Проверка сохранённых результатов

In [ ]:
import csv
RUN_NAMES = [
    'tanp_gating_film_lastfm_seed2020',
    'tanp_gating_film_movielens_seed2020',
    'tanp_gating_film_gowalla_seed2020',
]
final_rows = []
for run_name in RUN_NAMES:
    run_dir = DRIVE_ROOT / 'artifacts/runs' / run_name
    required = [
        run_dir / f'{run_name}.pt',
        run_dir / f'{run_name}.log',
        run_dir / f'{run_name}.config',
        run_dir / 'command.txt',
        run_dir / 'run_metadata.json',
        DRIVE_ROOT / 'results' / f'{run_name}_metrics.csv',
        DRIVE_ROOT / 'figures' / f'{run_name}_metrics.png',
        DRIVE_ROOT / 'figures' / f'{run_name}_metrics.pdf',
    ]
    missing = [str(path.relative_to(DRIVE_ROOT)) for path in required if not path.is_file()]
    assert not missing, f'{run_name}: отсутствуют файлы {missing}'
    log_path = run_dir / f'{run_name}.log'
    epochs = [line for line in log_path.read_text(encoding='utf-8').splitlines() if line and not line.startswith('#')]
    assert len(epochs) == 100, f'{run_name}: найдено {len(epochs)} эпох вместо 100'
    assert epochs[0].split()[0] == '0' and epochs[-1].split()[0] == '99'
    metrics_path = DRIVE_ROOT / 'results' / f'{run_name}_metrics.csv'
    with metrics_path.open(encoding='utf-8') as stream:
        rows = list(csv.DictReader(stream))
    final_rows.append({
        'run_name': run_name,
        'epochs': len(epochs),
        'NDCG@10': rows[-1]['ndcg10'],
        'MAP@10': rows[-1]['map10'],
        'Precision@10': rows[-1]['precision10'],
    })
summary_path = DRIVE_ROOT / 'results' / 'tanp_gating_film_seed2020_summary.csv'
assert summary_path.is_file(), f'Нет сводной таблицы: {summary_path}'
print('Все обязательные файлы найдены:')
for row in final_rows:
    print(row)
print('Сводная таблица:', summary_path)